## Read github repo

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

## Q1-How many lesson pages

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
len(documents)

72

## Q2-Indexing and searching

In [4]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [5]:
from minsearch import Index

index = Index(
    text_fields = ["content"],
    keyword_fields = ["filename"]
)

In [6]:
index.fit(documents)

In [22]:
index.search("How does the agentic loop keep calling the model until it stops?")[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

## Q3-RAG

In [8]:
from dotenv import load_dotenv
load_dotenv(override=True)

from ingest import load_faq_data, build_index
from rag_helper import RAGGithub
from openai import OpenAI
import os

In [9]:
llm_client = OpenAI()

assistant = RAGGithub(
    index=index,
    llm_client=llm_client,
    model="gpt-5.4-mini"
)

In [10]:
response = assistant.rag("How does the agentic loop keep calling the model until it stops?")
response.usage.input_tokens

7121

## Q4 - Chunking

In [11]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [12]:
len(chunks)

295

## Q5 - RAG with chunking

In [13]:
index = Index(
    text_fields = ["content"],
    keyword_fields = ["filename"]
)

In [14]:
index.fit(chunks)

In [15]:
assistant.index = index

In [16]:
response = assistant.rag("How does the agentic loop keep calling the model until it stops?")
response.usage.input_tokens

2304

## Q6-Turning it into an agent

In [17]:
from rag_helper import RAGGithubAgentic

agentic = RAGGithubAgentic(
    index=index
)
agent_response=agentic.rag("How does the agentic loop keep calling the model until it stops?")

In [20]:
# Count ToolCallItems in new_items
tool_calls = [item for item in agent_response.new_items if item.type == "tool_call_item"]
len(tool_calls)

3